# Restart an Interrupted Extraction

A batch extraction over a large corpus can run for hours. If the process stops, its Bedrock batch jobs keep running; run it again with the same `run_id` and it takes up where it stopped.

This notebook stops a run, restarts it, then builds and queries a graph. Restart covers batch extraction with a fixed batch size; `auto_tune=True` with a `run_id` raises a `ValueError`. See [Restarting an interrupted extraction](https://awslabs.github.io/graphrag-toolkit/lexical-graph/indexing/#restarting-an-interrupted-extraction).

## How restart works

A run keeps its records under `s3://<bucket>/restart-example/extracted/<collection_id>/_runs/<run_id>/`: `plan.json` holds the document ids in order, the batch size and the worker count, and `partitions/<partition_id>.json` names each partition's Bedrock job and attempt. A restart divides the documents as the plan did.

A document's id is derived from its text and metadata, so neither may change between runs. Documents are staged, each with a completion marker, after both extraction stages finish, so a run stopped during its first jobs has staged none.

On restart, a job still queued or running is waited on, a completed job has its output downloaded, and a failed, stopped or expired job is submitted again. A different `extraction_batch_size` or document set is refused with a `RunPlanMismatch`; use a new `run_id`.

## Prerequisites

Install the toolkit with the [Setup](./00-Setup.ipynb) notebook. This notebook needs lexical-graph 3.20 or later. Until 3.20 is on PyPI, install it from the `v3.20.x` branch:

```
!pip install "https://github.com/awslabs/graphrag-toolkit/archive/refs/heads/v3.20.x.zip#subdirectory=lexical-graph"
```

[Batch extraction](https://awslabs.github.io/graphrag-toolkit/lexical-graph/batch-extraction/) needs resources the lexical-graph templates do not create. Deploy [`graphrag-toolkit-bedrock-batch.json`](https://github.com/awslabs/graphrag-toolkit/tree/main/examples/lexical-graph/cloudformation-templates/graphrag-toolkit-bedrock-batch.json) in the same Region first. Then create or update the lexical-graph stack with its `BatchClientPolicyArn` output in `IamPolicyArn`, and add the other two outputs to your `.env` file:

```
export EXTRACTION_BUCKET=<BatchBucket output>
export BATCH_INFERENCE_ROLE=<BatchInferenceRole output>
```

`EXTRACTION_MODEL` must support [Bedrock batch inference](https://docs.aws.amazon.com/bedrock/latest/userguide/batch-inference-supported.html).

### Dataset

Bedrock batch inference needs at least 100 records in a job; with fewer there is no job to restart. This notebook uses the 100 aviation accident reports in [`ntsb-md`](https://github.com/awslabs/graphrag-toolkit/tree/main/examples/lexical-graph/workshop/assets/source-data/ntsb-md), from the [NTSB aviation investigation reports](https://www.ntsb.gov/investigations/AccidentReports/Pages/aviation.aspx). Their 1,100 chunks give each of the two workers below a job. The next cell downloads them.

In [ ]:
%%bash
if [ ! -d ntsb-md ]; then
  git clone --depth 1 --filter=blob:none --sparse https://github.com/awslabs/graphrag-toolkit.git graphrag-toolkit-src
  git -C graphrag-toolkit-src sparse-checkout set examples/lexical-graph/workshop/assets/source-data/ntsb-md
  cp -r graphrag-toolkit-src/examples/lexical-graph/workshop/assets/source-data/ntsb-md ./ntsb-md
fi
ls ntsb-md | wc -l

## Define the extraction

The first run is killed, so it runs from a file both runs share. `run_plan_store.staging_handler(run_id)` skips source documents already stored whole.

In [ ]:
%%writefile restart_extraction.py
import os

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, GraphRAGConfig, IndexingConfig, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.extract import BatchConfig, RunPlanStore

from llama_index.core.schema import Document

DOCS_DIR = 'ntsb-md'
KEY_PREFIX = 'restart-example'
EXTRACTION_BATCH_SIZE = 100


def run_plan_store(collection_id):
    return RunPlanStore(
        bucket_name=os.environ['EXTRACTION_BUCKET'],
        key_prefix=f'{KEY_PREFIX}/extracted',
        collection_id=collection_id
    )


def read_report(file_name):
    with open(os.path.join(DOCS_DIR, file_name)) as f:
        return Document(text=f.read(), metadata={'file_name': file_name})


def load_docs():
    return [read_report(name) for name in sorted(os.listdir(DOCS_DIR)) if name.endswith('.md')]


def extract(collection_id, run_id, batch_size=EXTRACTION_BATCH_SIZE):

    set_logging_config('INFO')

    GraphRAGConfig.extraction_batch_size = batch_size
    GraphRAGConfig.extraction_num_workers = 2

    store = run_plan_store(collection_id)

    indexing_config = IndexingConfig(
        batch_config=BatchConfig(
            region=GraphRAGConfig.aws_region,
            bucket_name=os.environ['EXTRACTION_BUCKET'],
            key_prefix=f'{KEY_PREFIX}/batch-inference',
            role_arn=os.environ['BATCH_INFERENCE_ROLE'],
            max_batch_size=1000,
            max_num_concurrent_batches=1
        )
    )

    with (
        GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE']) as graph_store,
        VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE']) as vector_store
    ):

        graph_index = LexicalGraphIndex(
            graph_store,
            vector_store,
            indexing_config=indexing_config
        )

        graph_index.extract(
            load_docs(),
            handler=store.staging_handler(run_id),
            show_progress=True,
            run_id=run_id,
            run_plan_store=store
        )

The next cell picks a collection id and a run id. The restart needs the same run id.

In [ ]:
%reload_ext dotenv
%dotenv

import os
import uuid

from graphrag_toolkit.lexical_graph import GraphRAGConfig, set_logging_config
from graphrag_toolkit.lexical_graph.indexing.extract.batch_extractor_base import LIVE_JOB_STATES

from restart_extraction import KEY_PREFIX, extract, run_plan_store

set_logging_config('INFO')

bucket_name = os.environ['EXTRACTION_BUCKET']
collection_id = f'ntsb-{uuid.uuid4().hex[:8]}'
run_id = f'run-{uuid.uuid4().hex[:8]}'

store = run_plan_store(collection_id)
manifest_store = store.manifest_store(run_id)


def jobs_recorded():
    records = manifest_store.read_partitions(GraphRAGConfig.s3)
    return {partition: record for partition, record in records.items() if record.job_arn}


def job_status(record):
    return GraphRAGConfig.bedrock.get_model_invocation_job(jobIdentifier=record.job_arn)['status']


print(f'collection_id: {collection_id}')
print(f'run_id       : {run_id}')

## Start a run and stop it

The first run starts in its own session, so killing the session also kills the workers. The next cell kills it once both partitions have a submitted Bedrock job, or after ten minutes. The jobs keep running.

In [ ]:
import signal
import subprocess
import sys
import time

with open('first-run.log', 'w') as log:
    first_run = subprocess.Popen(
        [
            sys.executable, '-c',
            'import sys; from restart_extraction import extract; extract(sys.argv[1], sys.argv[2])',
            collection_id, run_id
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

deadline = time.monotonic() + 600

try:
    while len(jobs_recorded()) < 2 and time.monotonic() < deadline:
        if first_run.poll() is not None:
            raise RuntimeError(f'The first run ended before it submitted its jobs. See first-run.log [exit code: {first_run.returncode}]')
        time.sleep(10)
finally:
    if first_run.poll() is None:
        os.killpg(first_run.pid, signal.SIGKILL)
    first_run.wait()

print(f'First run stopped with {len(jobs_recorded())} jobs recorded [exit code: {first_run.returncode}]')

## Inspect the stopped run

`plan_resume` reports what a restart will reuse from the run records and the collection. It submits nothing.

In [ ]:
from graphrag_toolkit.lexical_graph.indexing.extract.resume import plan_resume
from graphrag_toolkit.lexical_graph.indexing.load.s3_based_docs import is_completion_marker

plan = store.read(run_id, GraphRAGConfig.s3)
print(f'Plan: {len(plan.document_ids)} documents, batch_size: {plan.batch_size}, num_workers: {plan.num_workers}')
print()

interrupted = jobs_recorded()
for partition, record in interrupted.items():
    print(f'Partition {partition[:10]}  attempt: {record.attempt}  job: {record.job_name}  Bedrock status: {job_status(record)}')
print()

pages = GraphRAGConfig.s3.get_paginator('list_objects_v2').paginate(
    Bucket=bucket_name,
    Prefix=f'{store.key_prefix}/{collection_id}/'
)
keys = [obj['Key'] for page in pages for obj in page.get('Contents', [])]
for key in keys:
    print(key)
print(f'Completion markers: {len([key for key in keys if is_completion_marker(key)])}')
print()

report = plan_resume(manifest_store, GraphRAGConfig.s3)
print(report.describe())

## Restart with the same run id

The cell runs until every job finishes. `same job as the first run` means the restart reused the first run's job. Finished partition records are folded into `rollup.json`.

In [ ]:
extract(collection_id, run_id)

for partition, record in manifest_store.read_partitions(GraphRAGConfig.s3).items():
    before = interrupted.get(partition)
    if before is None:
        submitted_by = 'submitted by the restart'
    elif before.job_arn == record.job_arn:
        submitted_by = 'same job as the first run'
    else:
        submitted_by = 'submitted again by the restart'
    print(f'Partition {partition[:10]}  attempt: {record.attempt}  state: {record.state}  job: {record.job_name}  ({submitted_by})')

A changed `batch_size` is refused:

In [ ]:
from graphrag_toolkit.lexical_graph.indexing.extract import RunPlanMismatch

try:
    extract(collection_id, run_id, batch_size=50)
except RunPlanMismatch as e:
    print(f'Refused: {e}')

## Build and query

The build stage reads the staged collection like any `S3BasedDocs` collection and skips the run directory. The graph uses the `restart` tenant to stay apart from the other notebooks' graphs; see [Multi-Tenancy](./04-Multi-Tenancy.ipynb). For queries, see [Querying](https://awslabs.github.io/graphrag-toolkit/lexical-graph/querying/) and [Traversal-based search](https://awslabs.github.io/graphrag-toolkit/lexical-graph/traversal-based-search/).

In [ ]:
from graphrag_toolkit.lexical_graph import LexicalGraphIndex, LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import S3BasedDocs

docs = S3BasedDocs(
    region=GraphRAGConfig.aws_region,
    bucket_name=bucket_name,
    key_prefix=store.key_prefix,
    collection_id=collection_id
)

with (
    GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE']) as graph_store,
    VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE']) as vector_store
):

    graph_index = LexicalGraphIndex(
        graph_store,
        vector_store,
        tenant_id='restart'
    )

    graph_index.build(docs, show_progress=True)

    query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
        graph_store,
        vector_store,
        tenant_id='restart'
    )

    response = query_engine.query('What happened to the Bell 206 helicopter near Blair, Nebraska?')

print(response.response)

## Clean up

The next cell stops this run's Bedrock jobs that are still queued or running, then deletes the staged collection, run records and batch files under `restart-example/`. It keeps the `restart` tenant's graph and vector data, and the local files `ntsb-md`, `graphrag-toolkit-src`, `restart_extraction.py` and `first-run.log`.

In [ ]:
for record in manifest_store.read_partitions(GraphRAGConfig.s3).values():
    if record.job_arn and job_status(record) in LIVE_JOB_STATES:
        GraphRAGConfig.bedrock.stop_model_invocation_job(jobIdentifier=record.job_arn)
        print(f'Stopped {record.job_name}')

for prefix in [f'{store.key_prefix}/{collection_id}/', f'{KEY_PREFIX}/batch-inference/']:
    pages = GraphRAGConfig.s3.get_paginator('list_objects_v2').paginate(Bucket=bucket_name, Prefix=prefix)
    keys = [obj['Key'] for page in pages for obj in page.get('Contents', [])]
    errors = []
    for i in range(0, len(keys), 1000):
        response = GraphRAGConfig.s3.delete_objects(
            Bucket=bucket_name,
            Delete={'Objects': [{'Key': key} for key in keys[i:i + 1000]]}
        )
        errors.extend(response.get('Errors', []))
    for error in errors:
        print(f"Not deleted: {error['Key']} [{error['Code']}: {error['Message']}]")
    print(f'Deleted {len(keys) - len(errors)} of {len(keys)} objects under s3://{bucket_name}/{prefix}')